**Owner: Teddy**

## Part 5: Integrated Waste Management Assistant

In this section, we integrate the three models into a single assistant. It accepts **either** an image **or** a text description, classifies the waste, and returns recycling instructions grounded in city policy.

### 5.1 Design Integration Architecture

The assistant routes the input to the right classifier, then feeds the predicted category into the RAG generator. The clean function interfaces defined in Parts 2-4 are what let the three models snap together:

```
            +-- image --> classify_waste_image (Part 2, CNN) --+
  input ----+                                                  +--> category --> generate_recycling_instructions (Part 4, RAG) --> instructions + policy sources
            +-- text  --> classify_waste_description (Part 3) --+
```

Interfaces used:
- `classify_waste_image(path) -> category` (Part 2)
- `classify_waste_description_with_confidence(text) -> (category, confidence)` (Part 3)
- `generate_recycling_instructions(category) -> (instructions, policy_docs)` (Part 4)

The text path also uses the **confidence** from Part 3: when confidence is below `LOW_CONFIDENCE`, the assistant asks the resident for more detail instead of guessing (a simple user-feedback mechanism).

In [ ]:
# Part 2 did not export an image-prediction helper, so we define one here (uses the CNN `model` + `class_names`
# from Part 2). If Part 2 already defines classify_waste_image, that version is kept.
import numpy as np, tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as _mnv2_preprocess

if 'classify_waste_image' not in globals():
    def classify_waste_image(image_path, cnn=None):
        """Predict the waste category for an image file path using the Part 2 CNN."""
        cnn = cnn if cnn is not None else model          # the trained MobileNetV2 from Part 2
        img = tf.keras.utils.load_img(image_path, target_size=(224, 224))
        arr = tf.keras.utils.img_to_array(img)[None, ...]
        arr = _mnv2_preprocess(arr)                        # same [-1, 1] scaling used in training
        probs = cnn.predict(arr, verbose=0)[0]
        return class_names[int(np.argmax(probs))]

print('classify_waste_image ready')

### 5.2 Implement Integrated Assistant

In [ ]:
def waste_management_assistant(input_data, input_type="image"):
    """
    Unified EcoSort assistant.
      input_type="image": input_data is an image file path  -> CNN (Part 2)
      input_type="text" : input_data is a description string -> text model (Part 3)
    Classifies the item, then generates policy-grounded recycling instructions (Part 4).
    Returns a dict with the category, confidence (text path), instructions and policy sources.
    """
    try:
        confidence = None
        if input_type == "image":
            category = classify_waste_image(input_data)
        elif input_type == "text":
            category, confidence = classify_waste_description_with_confidence(input_data)
            # user-feedback mechanism: on low confidence, ask for more detail instead of guessing
            if confidence < LOW_CONFIDENCE:
                return {"status": "needs_more_detail",
                        "message": "The description is ambiguous - please add material, size or condition.",
                        "predicted_category": category, "confidence": round(float(confidence), 3)}
        else:
            return {"status": "error", "message": f"input_type must be 'image' or 'text', got '{input_type}'"}

        if category not in CATEGORIES:
            return {"status": "error", "message": f"Unrecognised category '{category}'"}

        instructions, policies = generate_recycling_instructions(category)
        sources = [p.get('source', str(p)) if isinstance(p, dict) else str(p) for p in policies]
        result = {"status": "ok", "input_type": input_type, "waste_category": category,
                  "recycling_instructions": instructions, "policy_sources": sources}
        if confidence is not None:
            result["confidence"] = round(float(confidence), 3)
        return result
    except FileNotFoundError:
        return {"status": "error", "message": f"Image file not found: {input_data}"}
    except Exception as e:
        return {"status": "error", "message": f"{type(e).__name__}: {e}"}

### 5.3 Evaluate the Integrated System

In [ ]:
# Text input
from pprint import pprint
pprint(waste_management_assistant("empty plastic water bottle with the cap on", input_type="text"))

In [ ]:
# Image input - use a real image from the test categories
import pathlib
true_cat = CATEGORIES[0]
img_path = str(next((pathlib.Path(IMAGE_DIR) / true_cat).glob('*')))
res = waste_management_assistant(img_path, input_type="image")
print('True folder :', true_cat)
print('Predicted   :', res.get('waste_category'))
print('Policies    :', res.get('policy_sources'))
print('\nInstructions:\n', res.get('recycling_instructions'))

In [ ]:
# A few more cases, including edge cases (ambiguous text, and a bad image path)
for txt in ["rotten banana peels and food scraps", "flattened cardboard box", "thing"]:
    print(txt, '->', waste_management_assistant(txt, input_type="text").get('status'),
          '|', waste_management_assistant(txt, input_type="text").get('waste_category',
                waste_management_assistant(txt, input_type="text").get('message')))
print('bad path ->', waste_management_assistant('no_such_file.jpg', input_type='image'))

### System evaluation notes

- The assistant cleanly routes image vs text input through the agreed interfaces and returns policy-grounded instructions in one call.
- Robustness: it validates `input_type`, guards against unknown categories, handles a missing image file, and catches unexpected errors so a single bad input never crashes the pipeline.
- The text path exposes a confidence score and defers to the user (asks for more detail) when the model is unsure, rather than returning a low-quality guess.
- As expected for a small generator, the RAG wording is simple; the key property is that it draws on the retrieved city policy for the predicted category.